# 01 - Hypothesis Tests, Power & Sample Size
**Milestones 1 & 2**

This notebook covers the foundations of experiment design:
* hypothesis tests, p-values and confidence intervals,
* statistical power and the minimum detectable effect (MDE),
* the sample-size / power calculator that powers the FastAPI service.

All statistics come from `statsmodels`, wrapped by `src/core/power.py`.


In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def _find_project_root(start):
    d = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(d, "src")) and os.path.exists(
            os.path.join(d, "src", "config.py")
        ):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return os.path.abspath(start)
        d = parent

PROJECT_ROOT = _find_project_root(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import artifact_path, raw_path, ensure_directories
ensure_directories()
FIGS = artifact_path("figures")
os.makedirs(FIGS, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("FIGURES    :", FIGS)


## 1. Sample size for a conversion (proportion) metric
Baseline conversion 10%, we want to detect a 10% relative lift with 80% power at alpha=0.05.

In [ ]:
from src.core import power as power_core
import pandas as pd

ss = power_core.sample_size_two_proportions(p_control=0.10, relative_lift=0.10, alpha=0.05, power=0.80)
pd.Series(ss)

### Power as a function of sample size
The same effect is much easier to detect with more users per arm.

In [ ]:
import numpy as np

sizes = np.arange(500, 25001, 500)
powers = [power_core.power_two_proportions(0.10, 0.11, int(n), alpha=0.05)["power"] for n in sizes]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sizes, powers, color="#4f46e5", lw=2)
ax.axhline(0.80, ls="--", color="#dc2626", label="target power = 0.80")
ax.set_xlabel("users per arm"); ax.set_ylabel("statistical power")
ax.set_title("Power curve - baseline 10%, +10% relative lift")
ax.legend(); fig.tight_layout()
fig.savefig(os.path.join(FIGS, "01_power_curve.png"), dpi=120, bbox_inches="tight");
plt.show()

## 2. Required sample size vs. the effect we want to detect
Smaller lifts need dramatically larger samples.

In [ ]:
lifts = np.array([0.02, 0.05, 0.10, 0.20, 0.30])
n_req = [power_core.sample_size_two_proportions(0.10, relative_lift=float(l))["n_total"] for l in lifts]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([f"{int(l*100)}%" for l in lifts], n_req, color="#0ea5e9")
ax.set_xlabel("relative lift to detect"); ax.set_ylabel("total sample size")
ax.set_title("Sample size grows fast as the effect shrinks")
for i, v in enumerate(n_req):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(FIGS, "01_sample_size_vs_lift.png"), dpi=120, bbox_inches="tight");
plt.show()

## 3. Minimum detectable effect (MDE)
With a fixed budget we can only detect effects larger than the MDE.

In [ ]:
for n in (1000, 5000, 20000):
    mde = power_core.mde_two_proportions(0.10, n_control=n)
    print(f"n/arm={n:>6,}  ->  MDE (Cohen's h)={mde['cohens_h']:.4f}"
          f"  ~ {mde['approx_absolute_lift']*100:.3f} pp absolute")

## 4. Continuous metrics (t-test)
For continuous outcomes we plan with Cohen's *d*.

In [ ]:
cont = power_core.sample_size_continuous(cohens_d=0.2, alpha=0.05, power=0.80)
print("sample size per arm for d=0.2:", cont["n_control"])
print("power at n=1000/arm, d=0.2     :", round(power_core.power_continuous(0.2, 1000)["power"], 3))
print("MDE (d) at n=1000/arm          :", round(power_core.mde_continuous(1000)["cohens_d"], 3))

## Takeaways
* Power rises with sample size and effect size, and falls with smaller alpha.
* Always size the experiment *before* launching; an underpowered test cannot be saved by luck.
* The same routines back the `/api/power/*` endpoints used by the React front end.